In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from example_models import get_upper_glycolysis
from mxlpy import KineticRunner, plot

# Metabolic control analysis

Metabolic control analysis answers the question: **what happens to the concentrations and fluxes if I slightly perturb the system?**  
It is thus a *local* measurement about which reactions hold the most control.  

The most common measurements are **elasticities** and **response coefficients**.  
The main difference between them is that *response coefficients* are calculated at **steady-state**, while *elasticities* can be calculated at every arbitrary state.


<div>
    <img src="assets/parameter-elasticity.png" 
         style="vertical-align:middle; max-height: 175px;" />
    <img src="assets/response-coefficient.png" 
         style="vertical-align:middle; max-height: 175px;" />
</div>

All the required functionality can be found in the `mxlpy.mca` module.

As an example, we will use the upper glycolysis model from the [Systems Biology textbook by Klipp et al. (2005)](https://www.wiley-vch.de/de/fachgebiete/naturwissenschaften/systems-biology-978-3-527-33636-4).

In [ ]:
from mxlpy import mca

### Variable elasticities

Variable elasticities are the sensitivity of reactions to a small change in the concentration of a variable.  
They are **not** a steady-state measurement and can be calculated for any arbitrary state.  

<img src="assets/variable-elasticity.png" style="max-width: 500px" />

Both the `concs` and `variables` arguments are optional.  
If `concs` is not supplied, the routine will use the initial conditions from the model.  
If `variables` is not supplied, the elasticities will be calculated for all variables.  

In [ ]:
elas = mca.variable_elasticities(
    get_upper_glycolysis(),
    to_scan=["GLC", "F6P"],
    variables={
        "GLC": 0.3,
        "G6P": 0.4,
        "F6P": 0.5,
        "FBP": 0.6,
        "ATP": 0.4,
        "ADP": 0.6,
    },
)

_ = plot.heatmap(elas)
plt.show()

### Parameter elasticities

Similarly, parameter elasticities are the sensitivity of reactions to a small change in the concentration of a variable.  
They are **not** a steady-state measurement and can be calculated for any arbitrary state.  

<img src="assets/parameter-elasticity.png" style="max-width: 500px" />

Both the `concs` and `parameters` arguments are optional.  
If `concs` is not supplied, the routine will use the initial conditions from the model.  
If `parameters` is not supplied, the elasticities will be calculated for all parameters.  

In [ ]:
elas = mca.parameter_elasticities(
    get_upper_glycolysis(),
    variables={
        "GLC": 0.3,
        "G6P": 0.4,
        "F6P": 0.5,
        "FBP": 0.6,
        "ATP": 0.4,
        "ADP": 0.6,
    },
    to_scan=["k1", "k2"],
)

_ = plot.heatmap(elas)
plt.show()

### Response coefficients

Response coefficients show the sensitivity of variables and reactions **at steady-state** to a small change in a parameter.  


<img src="assets/response-coefficient.png" style="max-width: 500px" />

If the parameter is proportional to the rate, they are also called **control coefficients**.

Calculation of these is run in parallel by default.  

In [ ]:
crcs, frcs = mca.response_coefficients(
    get_upper_glycolysis(),
    to_scan=["k1", "k2", "k3", "k4", "k5", "k6", "k7"],
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
_ = plot.heatmap(crcs, ax=ax1)
_ = plot.heatmap(frcs, ax=ax2)
plt.show()

### Time-dependent MCA

Response coefficients and elasticities aren't restricted to steady states.
Passing `time_points` gives them along the trajectory starting from the initial conditions (or `variables`): how every variable and flux **at every time point** responds to a change of a parameter (Ingalls & Sauro 2003).
They come from integrating the forward sensitivities alongside the model, i.e. a single simulation for all parameters.

The rows of the result are then indexed by `(time, output)`, so `rc.variables.loc[t]` gives all variable responses at time `t`, and `rc.for_parameter(name)` a time × output table ready for plotting.
Normalised coefficients are undefined where an output is zero, e.g. for variables starting at zero, which is why the time points below start just after 0.

In [ ]:
time_points = np.linspace(0.1, 10, 100)
rc = mca.response_coefficients(
    get_upper_glycolysis(),
    to_scan=["k1", "k2", "k7"],
    time_points=time_points,
)

fig, axs = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, name in zip(axs, ["k1", "k2", "k7"], strict=True):
    _ = plot.lines(rc.for_parameter(name).loc[:, ["GLC", "G6P", "F6P", "FBP", "ATP"]], ax=ax)
    ax.set(title=f"Response to {name}", xlabel="time", ylabel="Normalised response")
plt.show()

Elasticities along the trajectory work the same way.
They're computed exactly from the generated derivatives of the model at every simulated state, without further simulations.
The mass-action reactions of this model have constant normalised elasticities, but those of the reversible `v3` (G6P ⇌ F6P) change with its distance from equilibrium: they start at 1 (substrate) and 0 (product) while there is no F6P yet, and grow in magnitude as the reaction approaches equilibrium.

In [ ]:
elas = mca.variable_elasticities(get_upper_glycolysis(), time_points=time_points)

fig, ax = plt.subplots()
_ = plot.lines(elas.xs("v3", level=1).loc[:, ["G6P", "F6P"]], ax=ax)
ax.set(title="Elasticities of v3", xlabel="time", ylabel="Normalised elasticity")
plt.show()

### Steady-state stability

After finding a steady state you can classify its local stability from the eigenvalues of the Jacobian.

`steady_state_stability` evaluates the symbolic Jacobian at the given state, computes eigenvalues with `numpy.linalg.eigvals`, and returns a `SteadyStateStability` object:

| Field | Description |
|---|---|
| `eigenvalues` | Complex Jacobian eigenvalues |
| `spectral_abscissa` | Largest real part — negative ⟺ stable |
| `is_stable` | `True` when all real parts are negative |
| `has_oscillatory` | `True` when any eigenvalue has nonzero imaginary part |
| `classification` | `"stable node"` \| `"stable spiral"` \| `"unstable node"` \| `"unstable spiral"` \| `"saddle point"` \| `"centre"` \| `"non-hyperbolic"` |

In [ ]:
from mxlpy import Simulator
from mxlpy.mca import steady_state_stability

model = get_upper_glycolysis()
ss = (
    Simulator(model)
    .simulate_to_steady_state()
    .get_result()
    .unwrap_or_err()
    .get_new_y0()
)

stab = steady_state_stability(model, ss)
print(f"Classification : {stab.classification}")
print(f"Is stable      : {stab.is_stable}")
print(f"Spectral absc. : {stab.spectral_abscissa:.4f}")
print(f"Has oscillatory: {stab.has_oscillatory}")
print(f"Eigenvalues    : {stab.eigenvalues}")

### Supply-demand rate characteristics

Where the analyses above are *local* (a small perturbation around a single state), **rate characteristics** show the *global* picture: how the reactions that produce and consume a metabolite respond across its whole concentration range.

`rate_characteristics` clamps a chosen variable to a logarithmically spaced range around its steady-state concentration, recomputes the steady state of the remaining variables at each point, and collects the fluxes of the reactions that **produce** it (*supply*) and **consume** it (*demand*). The intersection of total supply and total demand is the steady state — plotting both curves places that operating point in the context of the full range.

Reactions are classified structurally from the stoichiometry: a positive coefficient marks a supply reaction, a negative one a demand reaction. The scan spans `[ss / min_factor, ss * max_factor]` with `n_points` log-spaced samples.

The result exposes:

| Attribute / method | Description |
|---|---|
| `supply_fluxes` / `demand_fluxes` | Per-reaction fluxes vs. the clamped concentration (one column per reaction) |
| `total_supply` / `total_demand` | Summed supply / demand flux |
| `steady_state_conc` | Concentration of the variable at the original steady state |
| `plot_supply_demand()` | Total supply and demand on a single log-log axis |
| `plot()` | Three-panel summary: total supply vs. demand, per-reaction supply, per-reaction demand |

Here we scan `F6P`, which is produced by two reactions (`v3`, `v5`) and consumed by one (`v4`).

In [ ]:
rc = mca.rate_characteristics(get_upper_glycolysis(), "F6P")

_ = rc.plot()
plt.show()

<div style="color: #ffffff; background-color: #04AA6D; padding: 3rem 1rem 3rem 1rem; box-sizing: border-box">
    <h2>First finish line</h2>
    With that you now know most of what you will need from a day-to-day basis about metabolic control analysis in mxlpy.
    <br />
    <br />
    Congratulations!
</div>

## Advanced concepts / customisation

### Normalisation

By default the elasticities and response coefficients are **normalised**, so e.g. for the response coefficients the following is calculated:

$$C^J_{v_i} = \left( \frac{dJ}{dp} \frac{p}{J} \right) \bigg/ \left( \frac{\partial v_i}{\partial  p}\frac{p}{v_i} \right)$$

You can also obtain the **non-normalised** coefficients, by setting `normalized=False`, which amounts to the following calculation:

$$C^J_{v_i} = \left( \frac{dJ}{dp} \frac{p}{J} \right)$$

In [ ]:
_ = mca.response_coefficients(
    get_upper_glycolysis(),
    to_scan=["k1", "k2", "k3", "k4", "k5", "k6", "k7"],
    normalized=False,
)

### Forward sensitivities and finite differences

We wrote above that we *slightly* change the value, but by how much exactly?
By default, not at all: `mxlpy` computes the response coefficients from the **forward sensitivities** of the steady state.
The implicit function theorem gives the derivatives of the steady state w.r.t. all parameters from a single steady-state simulation and one linear solve, $J \, \partial x^* / \partial p = -\partial f / \partial p$, with both derivatives generated symbolically from the model.
This is exact and needs far fewer simulations than perturbing every parameter.

Models that can't be differentiated symbolically (e.g. neural-network surrogates) fall back to **central finite differences**, changing every parameter to
$$\textrm{value} \cdot \left(1 \pm \Delta \right)$$
and simulating both steady states, in parallel.
How derivatives are computed is configured on the runner: `derivatives="finite_difference"` always uses finite differences, `displacement` sets the relative change $\Delta$ (default `1e-3`, i.e. `0.1 %`).

In [ ]:
_ = mca.response_coefficients(
    get_upper_glycolysis(),
    to_scan=["k1", "k2", "k3", "k4", "k5", "k6", "k7"],
    runner=KineticRunner(derivatives="finite_difference", displacement=1e-2),
)